# Craftax RL Fine-Tuning Ablations

Mirrors the MiniHack ablations notebook exactly, adapted for Craftax/JAX.

**Four ablations:**

| # | Ablation | Rules out |
|---|----------|-----------|
| 1 | KL Penalty | Catastrophic forgetting / updates too large |
| 2 | Frozen Backbone | Deep gradient flow destabilising representations |
| 3 | BC on Wins | ELBO t-marginalisation as the specific cause |
| 4 | Low-t Only (t ∈ [ε, 0.2]) | High-t gradient dominance |

Baseline RL result (from earlier run): ~2/226 score, collapsed from pretrained.

**Structure:** Python for-loop outer training (flexibility), JAX-jitted inner grad step and rollout.

## 0. Setup

In [ ]:
import os
os.environ["XLA_FLAGS"] = os.environ.get("XLA_FLAGS", "") + " --xla_gpu_enable_command_buffer="

import sys
import pathlib
import copy
import time
import json
import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
import optax

# Add repo to path — adjust if needed
REPO = pathlib.Path(".").resolve()
if not (REPO / "src").exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "Craftax_Baselines"))

print(f"JAX backend: {jax.default_backend()} | Devices: {jax.devices()}")
print(f"REPO: {REPO}")

## 1. Config

In [ ]:
import yaml

# Load base config from project defaults; keep only ablation-specific overrides here.
with open("../configs/defaults.yaml") as _f:
    cfg = yaml.safe_load(_f)

with open("../configs/ablations.yaml") as _f:
    abl_cfg_base = yaml.safe_load(_f)

# ── Paths (not in defaults.yaml) ────────────────────────────────────────────
PRETRAINED_CKPT_DIR = "/home/jovyan/craftax-ReMDM-planner/checkpoints_offline/policies/Craftax-Classic-Symbolic-v1-OfflineDiffusion-1000M"
PPO_CKPT_PATH       = "/home/jovyan/craftax-ReMDM-planner/checkpoints/policies/Craftax-Classic-Symbolic-v1-PPO_RNN-1000M"

# W&B logging disabled for ablation notebook runs.
# TODO: wire up via src/planners/logging.py if W&B metrics are needed.
USE_WANDB = False

# ── Ablation-specific run config (not in defaults.yaml) ─────────────────────
MAX_ITER    = 500
EVAL_EVERY  = 100
EVAL_STEPS  = 512
EVAL_REPLAN = 4    # env steps per diffusion plan during eval

# ── Read everything else from defaults.yaml ─────────────────────────────────
PLAN_HORIZON        = cfg["plan_horizon"]
NUM_ENVS            = cfg["num_envs"]
NUM_STEPS           = cfg["num_steps"]
LR                  = cfg["lr"]
MAX_GRAD_NORM       = cfg["max_grad_norm"]
BATCH_SIZE          = cfg["batch_size"]
COLLECT_TEMP        = cfg["collect_temperature"]
RETURN_WEIGHT_CAP   = cfg["return_weight_cap"]
RETURN_WEIGHT_FLOOR = 0.1  # not in defaults.yaml; kept local

# Ablation hyperparameters from ablations.yaml
KL_COEF       = abl_cfg_base.get("kl_coef", 0.1)
T_MAX_LOW     = abl_cfg_base.get("t_max_low", 0.2)
WIN_THRESHOLD = abl_cfg_base.get("win_threshold", 0.0)

# Model architecture
D_MODEL         = cfg["d_model"]
N_HEADS         = cfg["n_heads"]
N_LAYERS        = cfg["n_layers"]
D_FF            = cfg["d_ff"]
OBS_ENC_LAYERS  = cfg["obs_encoder_layers"]
OBS_ENC_WIDTH   = cfg["obs_encoder_width"]
DROPOUT_RATE    = cfg["dropout_rate"]
DIFFUSION_SCHED = cfg["diffusion_schedule"]
TRAIN_SIGMA     = cfg["train_sigma"]
LABEL_SMOOTHING = cfg["label_smoothing"]
PPO_MODEL_TYPE  = cfg["ppo_model_type"]
LAYER_SIZE      = cfg["layer_size"]

# Build config dict for make_env / build_model (uppercase keys expected).
config = {
    "ENV_NAME":              cfg["env_name"],
    "USE_OPTIMISTIC_RESETS": cfg["use_optimistic_resets"],
    "NUM_ENVS":              NUM_ENVS,
    "NUM_STEPS":             NUM_STEPS,
    "PLAN_HORIZON":          PLAN_HORIZON,
    "DIFFUSION_SCHEDULE":    DIFFUSION_SCHED,
    "D_MODEL":               D_MODEL,
    "N_HEADS":               N_HEADS,
    "N_LAYERS":              N_LAYERS,
    "D_FF":                  D_FF,
    "OBS_ENCODER_LAYERS":    OBS_ENC_LAYERS,
    "OBS_ENCODER_WIDTH":     OBS_ENC_WIDTH,
    "DROPOUT_RATE":          DROPOUT_RATE,
    "PPO_MODEL_TYPE":        PPO_MODEL_TYPE,
    "LAYER_SIZE":            LAYER_SIZE,
    "PPO_CHECKPOINT_PATH":   PPO_CKPT_PATH,
    "COLLECT_TEMPERATURE":   COLLECT_TEMP,
    "LR":                    LR,
    "MAX_GRAD_NORM":         MAX_GRAD_NORM,
    "TRAIN_SIGMA":           TRAIN_SIGMA,
    "LABEL_SMOOTHING":       LABEL_SMOOTHING,
    "REMASK_STRATEGY":       cfg["remask_strategy"],
    "ETA":                   cfg["eta"],
    "USE_LOOP":              cfg["use_loop"],
    "T_ON":                  cfg["t_on"],
    "T_OFF":                 cfg["t_off"],
    "TEMPERATURE":           cfg["temperature"],
    "TOP_P":                 cfg["top_p"],
    "VAL_DIFFUSION_STEPS":   cfg["val_diffusion_steps"],
}

print(f"Config ready | MAX_ITER={MAX_ITER} | EVAL_EVERY={EVAL_EVERY}")


## 2. Environment, Model, PPO

In [ ]:
from src.planners.env import make_env, Transition
from src.planners.model import build_model, init_params, create_train_state, make_apply_fns
from src.planners.ppo import PPOAgent, build_ppo_network, load_ppo_params
from src.planners.common import make_grad_step
from src.diffusion.loss import compute_loss
from src.diffusion.schedules import SCHEDULE_MAP
from src.diffusion.forward import forward_process
from src.diffusion.sampling import sample_plan

# Environment
env, env_params = make_env(config, NUM_ENVS)
num_actions = env.action_space(env_params).n
obs_shape   = env.observation_space(env_params).shape
obs_dim     = obs_shape[0]
print(f"obs_dim={obs_dim} | num_actions={num_actions}")

# Noise schedule
schedule_fn, schedule_deriv_fn = SCHEDULE_MAP[DIFFUSION_SCHED]

# Diffusion model
net = build_model(config, num_actions)
apply_eval, apply_train = make_apply_fns(net)

# PPO collector
ppo_net    = build_ppo_network(PPO_MODEL_TYPE, num_actions, LAYER_SIZE, config)
ppo_params = load_ppo_params(PPO_CKPT_PATH, ppo_net, PPO_MODEL_TYPE, NUM_ENVS, obs_shape, LAYER_SIZE)
ppo        = PPOAgent(ppo_net, ppo_params, PPO_MODEL_TYPE, LAYER_SIZE)

print("Environment, model, and PPO ready.")

In [ ]:
from src.planners.model import load_checkpoint

pretrained_params = load_checkpoint(
    net,
    jax.random.PRNGKey(0),
    obs_dim,
    PLAN_HORIZON,
    PRETRAINED_CKPT_DIR,
)
print(f"Loaded | leaves: {len(jax.tree.leaves(pretrained_params))}")

## 3. Shared Infrastructure

In [ ]:
# ---------------------------------------------------------------------------
# JIT-compiled rollout: collect NUM_STEPS of PPO data, extract windows
# ---------------------------------------------------------------------------
windows_per_env = NUM_STEPS - PLAN_HORIZON + 1

@jax.jit
def collect_rollout(env_state, obs, done, hstate, rng):
    """Run one PPO rollout. Returns flattened (obs, acts, valid, returns) windows."""
    def _env_step(carry, _):
        es, ob, dn, hs, rng = carry
        rng, act_rng, step_rng = jax.random.split(rng, 3)
        action, new_hs = ppo.act(ob, dn, hs, act_rng, temperature=COLLECT_TEMP)
        new_obs, es, reward, new_done, info = env.step(step_rng, es, action, env_params)
        t = Transition(done=dn, action=action, reward=reward, obs=ob, info=info)
        return (es, new_obs, new_done, new_hs, rng), t

    (env_state, obs, done, hstate, rng), traj = jax.lax.scan(
        _env_step, (env_state, obs, done, hstate, rng), None, NUM_STEPS,
    )

    def _window(t_idx):
        obs_t = traj.obs[t_idx]
        acts  = jax.lax.dynamic_slice(traj.action, (t_idx, 0), (PLAN_HORIZON, NUM_ENVS))
        dones = jax.lax.dynamic_slice(traj.done,   (t_idx + 1, 0), (PLAN_HORIZON - 1, NUM_ENVS))
        valid = ~jnp.any(dones, axis=0)
        rews  = jax.lax.dynamic_slice(traj.reward, (t_idx, 0), (PLAN_HORIZON, NUM_ENVS))
        window_return = jnp.sum(rews, axis=0)
        return obs_t, jnp.swapaxes(acts, 0, 1), valid, window_return

    obs_w, act_w, valid_w, ret_w = jax.vmap(_window)(jnp.arange(windows_per_env))

    flat_obs     = obs_w.reshape(-1, obs_dim)
    flat_acts    = act_w.reshape(-1, PLAN_HORIZON)
    flat_valid   = valid_w.reshape(-1)
    flat_returns = ret_w.reshape(-1)

    info_returned = traj.info["returned_episode"]
    env_score = jax.tree.map(
        lambda x: (x * info_returned).sum() / (info_returned.sum() + 1e-8),
        traj.info,
    )

    return env_state, obs, done, hstate, rng, flat_obs, flat_acts, flat_valid, flat_returns, env_score


@jax.jit
def eval_policy(params, rng):
    """Quick eval: run sample_plan + env for EVAL_STEPS steps, return score."""
    n_cycles = EVAL_STEPS // EVAL_REPLAN
    rng, env_rng = jax.random.split(rng)
    val_obs, val_es = env.reset(env_rng, env_params)

    def _cycle(carry, _):
        es, vo, rng = carry
        rng, p_rng = jax.random.split(rng)
        plan = sample_plan(
            apply_eval, params, p_rng, vo,
            num_actions, PLAN_HORIZON,
            num_steps=config["VAL_DIFFUSION_STEPS"],
            schedule_fn=schedule_fn,
            remask_strategy=config["REMASK_STRATEGY"],
            eta=config["ETA"],
            use_loop=config["USE_LOOP"],
            t_on=config["T_ON"],
            t_off=config["T_OFF"],
            temperature=config["TEMPERATURE"],
            top_p=config["TOP_P"],
        )
        def _step(c, step_i):
            es_i, vo_i, r = c
            r, s_rng = jax.random.split(r)
            vo_next, es_next, _, _, info = env.step(s_rng, es_i, plan[:, step_i], env_params)
            return (es_next, vo_next, r), info
        (es, vo, rng), infos = jax.lax.scan(_step, (es, vo, rng), jnp.arange(EVAL_REPLAN))
        return (es, vo, rng), infos

    _, cycle_infos = jax.lax.scan(_cycle, (val_es, val_obs, rng), None, n_cycles)
    infos  = jax.tree.map(lambda x: x.reshape(-1, *x.shape[2:]), cycle_infos)
    ret    = infos["returned_episode"]
    score  = jax.tree.map(lambda x: (x * ret).sum() / (ret.sum() + 1e-8), infos)
    return score

print(f"Rollout: {windows_per_env} windows/rollout | {NUM_ENVS * windows_per_env} samples/rollout")

In [ ]:
# ---------------------------------------------------------------------------
# Loss functions — all follow the ablations.md contract:
#   loss_fn(params, rng, acts, obs, valid, advantages) -> scalar
# _base_loss replaced by compute_loss (src/diffusion/loss.py, already imported).
# t_min/t_max kwarg restricts t-sampling range (added to compute_loss for ablations).
# ---------------------------------------------------------------------------

# 1. Baseline RL
def make_loss_baseline(apply_fn):
    """Return-weighted MDLM ELBO, full t range."""
    def loss(params, rng, acts, obs, valid, advantages):
        return compute_loss(apply_fn, params, rng, acts, obs, valid,
            num_actions, schedule_fn, schedule_deriv_fn,
            sigma_t=TRAIN_SIGMA, label_smoothing=LABEL_SMOOTHING,
            advantages=advantages)[0]
    return loss


# 2. KL Penalty
def make_loss_kl(apply_fn, ref_params, kl_coef=KL_COEF):
    """Return-weighted ELBO + KL(current || ref) penalty.
    FIX: split rng before both passes so noise is independent (was reusing same rng).
    """
    def loss(params, rng, acts, obs, valid, advantages):
        rng_rl, rng_kl = jax.random.split(rng)
        rl = compute_loss(apply_fn, params, rng_rl, acts, obs, valid,
            num_actions, schedule_fn, schedule_deriv_fn,
            sigma_t=TRAIN_SIGMA, label_smoothing=LABEL_SMOOTHING,
            advantages=advantages)[0]

        rng_kl, t_rng, mask_rng, drop_rng = jax.random.split(rng_kl, 4)
        B = acts.shape[0]
        t = jax.random.uniform(t_rng, (B,), minval=1e-5, maxval=1.0)
        alpha_t = schedule_fn(t)
        z_t = forward_process(mask_rng, acts, alpha_t, num_actions)
        is_masked = (z_t == num_actions).astype(jnp.float32)
        valid_m   = is_masked * valid[:, None].astype(jnp.float32)
        cur_log = jax.nn.log_softmax(apply_fn(params, obs, z_t, t, drop_rng), axis=-1)
        ref_log = jax.nn.log_softmax(
            apply_fn(jax.lax.stop_gradient(ref_params), obs, z_t, t, drop_rng), axis=-1)
        cur_prob = jnp.exp(cur_log)
        kl = (cur_prob * (cur_log - ref_log)).sum(-1)
        kl_mean = (kl * valid_m).sum(-1) / jnp.maximum(valid_m.sum(-1), 1.0)
        return rl + kl_coef * kl_mean.mean()
    return loss


# 3. BC on Wins
def make_loss_bc_wins(apply_fn):
    """Uniform ELBO — advantages ignored."""
    def loss(params, rng, acts, obs, valid, advantages):
        return compute_loss(apply_fn, params, rng, acts, obs, valid,
            num_actions, schedule_fn, schedule_deriv_fn,
            sigma_t=TRAIN_SIGMA, label_smoothing=LABEL_SMOOTHING,
            advantages=None)[0]
    return loss


# 4. Low-t Only
def make_loss_low_t(apply_fn, t_max=T_MAX_LOW):
    """Return-weighted ELBO restricted to t ∈ [eps, t_max]."""
    def loss(params, rng, acts, obs, valid, advantages):
        return compute_loss(apply_fn, params, rng, acts, obs, valid,
            num_actions, schedule_fn, schedule_deriv_fn,
            sigma_t=TRAIN_SIGMA, label_smoothing=LABEL_SMOOTHING,
            advantages=advantages, t_max=t_max)[0]
    return loss


print(f"Loss functions defined. KL_COEF={KL_COEF} | T_MAX_LOW={T_MAX_LOW}")


In [ ]:
# ---------------------------------------------------------------------------
# Generic grad step factory
# Separate from common.make_grad_step because: (1) supports pluggable loss_fn
# (KL, EWC, trust region, etc.) instead of hardwired compute_loss; (2) adds
# frozen_backbone gradient zeroing. These two differences justify a separate impl.
# ---------------------------------------------------------------------------

def make_ablation_grad_step(loss_fn, frozen_backbone=False):
    """Return a jitted grad step for the given loss function.

    Args:
        loss_fn:         Callable: (params, rng, acts, obs, valid, advantages) -> scalar.
        frozen_backbone: Zero out all gradients except the output head.
    """
    def _step(state, acts, obs, valid, rng, advantages):
        def _loss(params):
            # Follows ablations.md contract: (params, rng, acts, obs, valid, adv).
            # src/ablations/losses.py returns (scalar, info_dict); unwrap to scalar.
            result = loss_fn(params, rng, acts, obs, valid, advantages)
            return result[0] if isinstance(result, tuple) else result

        loss_val, grads = jax.value_and_grad(_loss)(state.params)

        if frozen_backbone:
            # Dense_5 is the Flax-assigned name for the output projection when
            # obs_encoder_layers=2: Dense_0/1=obs_enc, Dense_2=obs_tok,
            # Dense_3/4=time_emb, Dense_5=output_head(num_actions).
            # Verify with: sorted(pretrained_params['params'].keys())
            _HEAD_KEY = "Dense_5"
            def _mask_grad(path, grad):
                path_str = '/'.join(str(p.key) for p in path)
                return grad if _HEAD_KEY in path_str else jnp.zeros_like(grad)
            grads = jax.tree_util.tree_map_with_path(_mask_grad, grads)

        state = state.apply_gradients(grads=grads)
        return state, loss_val

    return jax.jit(_step)


def compute_return_weights(flat_returns, wins_only=False,
                           win_threshold=WIN_THRESHOLD,
                           cap=RETURN_WEIGHT_CAP, floor=RETURN_WEIGHT_FLOOR):
    """Normalise returns to per-sample advantage weights."""
    if wins_only:
        return (flat_returns > win_threshold).astype(jnp.float32)
    clipped = jnp.clip(flat_returns, 0.0, None)
    weights  = clipped / (jnp.mean(clipped) + 1e-8)
    return jnp.clip(weights, floor, cap)


print("Grad step factory defined.")


In [ ]:
# ---------------------------------------------------------------------------
# Enhanced ablation training loop
# Tracks: loss, eval score, gradient alignment, representation drift, t-distribution
# ---------------------------------------------------------------------------

@jax.jit
def compute_grad_alignment(params, ref_params, acts, obs, valid, rng, advantages):
    """Cosine similarity between RL gradient and oracle BC gradient.
    Oracle BC = uniform loss on the same batch (what supervised training would do).
    If cos_sim < 0: RL gradient actively points AWAY from improvement.
    If cos_sim ~ 0: RL gradient is noise.
    If cos_sim > 0: RL gradient points in a useful direction.
    """
    rng_rl, rng_bc = jax.random.split(rng)

    # RL gradient: return-weighted ELBO
    def rl_loss(p):
        return _base_loss(apply_train, p, rng_rl, acts, obs, valid, advantages)

    # Oracle BC gradient: uniform loss (supervised signal)
    def bc_loss(p):
        return _base_loss(apply_train, p, rng_bc, acts, obs, valid, advantages=None)

    rl_grads = jax.grad(rl_loss)(params)
    # FIX: evaluate BC gradient at current params (not ref_params) so both
    # gradients are at the same point for geometrically meaningful cosine sim.
    bc_grads = jax.grad(bc_loss)(params)

    rl_flat = jnp.concatenate([g.ravel() for g in jax.tree.leaves(rl_grads)])
    bc_flat = jnp.concatenate([g.ravel() for g in jax.tree.leaves(bc_grads)])

    cos_sim = jnp.dot(rl_flat, bc_flat) / (
        jnp.linalg.norm(rl_flat) * jnp.linalg.norm(bc_flat) + 1e-10
    )
    rl_norm = jnp.linalg.norm(rl_flat)
    bc_norm = jnp.linalg.norm(bc_flat)
    return cos_sim, rl_norm, bc_norm


@jax.jit
def compute_repr_drift(params, ref_params, obs, acts, rng):
    """KL divergence between current and pretrained model predictions.
    Measures how much the model has drifted from its pretrained state.
    High drift + performance collapse = representations corrupted.
    """
    B = obs.shape[0]
    rng, t_rng, mask_rng = jax.random.split(rng, 3)
    t = jax.random.uniform(t_rng, (B,), minval=0.3, maxval=0.7)  # mid-range t
    alpha_t = schedule_fn(t)
    z_t = forward_process(mask_rng, acts, alpha_t, num_actions)

    cur_logits = apply_eval(params,     obs, z_t, t)
    ref_logits = apply_eval(ref_params, obs, z_t, t)

    cur_log  = jax.nn.log_softmax(cur_logits, axis=-1)
    ref_log  = jax.nn.log_softmax(ref_logits, axis=-1)
    ref_prob = jnp.exp(ref_log)

    # KL(ref || cur): how much current differs from reference
    kl = (ref_prob * (ref_log - cur_log)).sum(-1).mean()
    return kl


@jax.jit
def compute_t_gradient_analysis(params, acts, obs, valid, advantages, rng):
    """Compare gradient norm contribution from high-t vs low-t samples.
    If high-t dominates and those gradients point in wrong direction,
    this directly supports the bias hypothesis.
    """
    rng_lo, rng_hi = jax.random.split(rng)

    def loss_low(p):
        return _base_loss(apply_train, p, rng_lo, acts, obs, valid, advantages,
                          t_min=_EPS, t_max=0.2)

    def loss_high(p):
        return _base_loss(apply_train, p, rng_hi, acts, obs, valid, advantages,
                          t_min=0.8, t_max=1.0)

    grads_low  = jax.grad(loss_low)(params)
    grads_high = jax.grad(loss_high)(params)

    norm_low  = jnp.linalg.norm(jnp.concatenate([g.ravel() for g in jax.tree.leaves(grads_low)]))
    norm_high = jnp.linalg.norm(jnp.concatenate([g.ravel() for g in jax.tree.leaves(grads_high)]))

    # Also compute alignment between low-t and high-t gradients
    low_flat  = jnp.concatenate([g.ravel() for g in jax.tree.leaves(grads_low)])
    high_flat = jnp.concatenate([g.ravel() for g in jax.tree.leaves(grads_high)])
    cos_sim   = jnp.dot(low_flat, high_flat) / (norm_low * norm_high + 1e-10)

    return norm_low, norm_high, cos_sim


def make_empty_history():
    return {
        'iter': [], 'loss': [],
        'env_score_iter': [], 'env_score': [],
        'eval_iter': [], 'eval_score': [],
        'grad_align_iter': [], 'grad_align': [], 'rl_grad_norm': [], 'bc_grad_norm': [],
        'repr_drift_iter': [], 'repr_drift': [],
        't_analysis_iter': [], 'norm_low_t': [], 'norm_high_t': [], 'lowhigh_cos': [],
    }

GRAD_ALIGN_EVERY = 25   # compute gradient alignment every N iters (expensive)
T_ANALYSIS_EVERY = 25


def run_ablation(name, params_init, loss_fn, frozen_backbone=False, wins_only=False):
    print(f'\n{"="*60}')
    print(f'ABLATION: {name}')
    print(f'{"="*60}')

    state     = create_train_state(net, params_init, LR, MAX_GRAD_NORM)
    grad_step = make_ablation_grad_step(loss_fn, frozen_backbone=frozen_backbone)
    history   = make_empty_history()

    rng = jax.random.PRNGKey(42)
    rng, env_rng = jax.random.split(rng)
    obs, env_state = env.reset(env_rng, env_params)
    done   = jnp.zeros(NUM_ENVS, dtype=bool)
    hstate = ppo.init_hidden(NUM_ENVS)

    running_loss = running_score = n_log = 0.0

    for iteration in range(1, MAX_ITER + 1):
        rng, rollout_rng = jax.random.split(rng)
        env_state, obs, done, hstate, rng, \
        flat_obs, flat_acts, flat_valid, flat_returns, env_score = \
            collect_rollout(env_state, obs, done, hstate, rollout_rng)

        advantages = compute_return_weights(flat_returns, wins_only=wins_only)

        n_samples = flat_obs.shape[0]
        rng, perm_rng, loss_rng, align_rng, drift_rng, t_rng = jax.random.split(rng, 6)
        perm       = jax.random.permutation(perm_rng, n_samples)
        flat_obs   = flat_obs[perm]
        flat_acts  = flat_acts[perm]
        flat_valid = flat_valid[perm]
        advantages = advantages[perm]

        obs_b  = flat_obs[:BATCH_SIZE]
        act_b  = flat_acts[:BATCH_SIZE]
        val_b  = flat_valid[:BATCH_SIZE]
        adv_b  = advantages[:BATCH_SIZE]

        state, loss_val = grad_step(state, act_b, obs_b, val_b, loss_rng, adv_b)

        running_loss  += float(loss_val)
        running_score += float(env_score.get('returned_episode_returns', jnp.array(0.0)))
        n_log += 1

        # --- Gradient alignment ---
        if iteration % GRAD_ALIGN_EVERY == 0:
            cos_sim, rl_norm, bc_norm = compute_grad_alignment(
                state.params, pretrained_params, act_b, obs_b, val_b, align_rng, adv_b
            )
            history['grad_align_iter'].append(iteration)
            history['grad_align'].append(float(cos_sim))
            history['rl_grad_norm'].append(float(rl_norm))
            history['bc_grad_norm'].append(float(bc_norm))
            print(f'  [{name}] iter {iteration} | grad_align={float(cos_sim):+.4f} | ' +
                  f'rl_norm={float(rl_norm):.4f} | bc_norm={float(bc_norm):.4f}')

        # --- Representation drift ---
        if iteration % GRAD_ALIGN_EVERY == 0:
            drift = compute_repr_drift(state.params, pretrained_params, obs_b, act_b, drift_rng)
            history['repr_drift_iter'].append(iteration)
            history['repr_drift'].append(float(drift))
            print(f'  [{name}] iter {iteration} | repr_drift (KL)={float(drift):.6f}')

        # --- t-distribution analysis ---
        if iteration % T_ANALYSIS_EVERY == 0:
            norm_lo, norm_hi, lo_hi_cos = compute_t_gradient_analysis(
                state.params, act_b, obs_b, val_b, adv_b, t_rng
            )
            history['t_analysis_iter'].append(iteration)
            history['norm_low_t'].append(float(norm_lo))
            history['norm_high_t'].append(float(norm_hi))
            history['lowhigh_cos'].append(float(lo_hi_cos))

        if iteration % 10 == 0:
            ml = running_loss  / max(n_log, 1)
            ms = running_score / max(n_log, 1)
            history['iter'].append(iteration)
            history['loss'].append(ml)
            history['env_score_iter'].append(iteration)
            history['env_score'].append(ms)
            wins_in_buf = int(jnp.sum(flat_returns > WIN_THRESHOLD))
            print(f'  [{name}] iter {iteration}/{MAX_ITER} | loss={ml:.4f} | ' +
                  f'score={ms:.3f} | wins={wins_in_buf}')
            running_loss = running_score = n_log = 0

        if iteration % EVAL_EVERY == 0:
            rng, eval_rng = jax.random.split(rng)
            eval_info  = eval_policy(state.params, eval_rng)
            eval_score = float(eval_info.get('returned_episode_returns', jnp.array(0.0)))
            history['eval_iter'].append(iteration)
            history['eval_score'].append(eval_score)
            print(f'  [{name}] Eval score: {eval_score:.4f}')

    rng, eval_rng = jax.random.split(rng)
    final_eval  = eval_policy(state.params, eval_rng)
    final_score = float(final_eval.get('returned_episode_returns', jnp.array(0.0)))
    print(f'  [{name}] FINAL score: {final_score:.4f}')
    return history, final_score, state.params


print("Enhanced training loop defined.")


## 4. Baseline Evaluation (Pretrained, No Fine-Tuning)

In [ ]:
print("Evaluating pretrained model (no fine-tuning)...")
rng = jax.random.PRNGKey(0)
rng, eval_rng = jax.random.split(rng)
baseline_info  = eval_policy(pretrained_params, eval_rng)
baseline_score = float(baseline_info.get('returned_episode_returns', jnp.array(0.0)))
print(f"Pretrained baseline score: {baseline_score:.4f}")

## 5. Ablation 0: Baseline RL (Return-Weighted ELBO)

The standard return-weighted ELBO with no modifications. This is the method that already
collapsed in earlier experiments. Running it here through the same loop so gradient alignment
and representation drift are tracked for direct comparison with the other ablations.

In [ ]:
loss_baseline_rl = make_loss_baseline(apply_train)
history_baseline_rl, score_baseline_rl, _ = run_ablation(
    name='Baseline-RL',
    params_init=jax.tree.map(jnp.array, pretrained_params),
    loss_fn=loss_baseline_rl,
    frozen_backbone=False,
    wins_only=False,
)
print(f"Baseline-RL final score: {score_baseline_rl:.4f} (pretrained: {baseline_score:.4f})")
# Baseline RL result — update with your actual result
BASELINE_RL_SCORE = score_baseline_rl

## 6. Ablation 1: KL Penalty

Return-weighted ELBO + KL divergence penalty against frozen pretrained model.
**Rules out:** Catastrophic forgetting / updates too large.

In [ ]:
loss_kl = make_loss_kl(apply_train, pretrained_params)
history_kl, score_kl, _ = run_ablation(
    name='KL-Penalty',
    params_init=jax.tree.map(jnp.array, pretrained_params),
    loss_fn=loss_kl,
    frozen_backbone=False,
    wins_only=False,
)
print(f"KL-Penalty final score: {score_kl:.4f} (baseline: {baseline_score:.4f})")

## 7. Ablation 2: Frozen Backbone

Only the output head (Dense_5) receives gradient updates.
**Rules out:** Deep gradient flow destabilising representations.

In [ ]:
loss_baseline = make_loss_baseline(apply_train)
history_frozen, score_frozen, _ = run_ablation(
    name='Frozen-Backbone',
    params_init=jax.tree.map(jnp.array, pretrained_params),
    loss_fn=loss_baseline,
    frozen_backbone=True,   # <-- zero out non-head gradients
    wins_only=False,
)
print(f"Frozen-Backbone final score: {score_frozen:.4f} (baseline: {baseline_score:.4f})")

## 8. Ablation 3: BC on Wins

Uniform loss on winning trajectories only. No return weighting, no ELBO difference.
**Isolates:** If this collapses, self-generated data distribution is the root cause.

In [ ]:
loss_bc = make_loss_bc_wins(apply_train)
history_bc, score_bc, _ = run_ablation(
    name='BC-on-Wins',
    params_init=jax.tree.map(jnp.array, pretrained_params),
    loss_fn=loss_bc,
    frozen_backbone=False,
    wins_only=True,   # <-- only use winning windows
)
print(f"BC-on-Wins final score: {score_bc:.4f} (baseline: {baseline_score:.4f})")

## 9. Ablation 4: Low-t Only

Return-weighted ELBO restricted to t ∈ [ε, 0.2].
**Tests:** High-t gradient bias hypothesis.

In [ ]:
loss_lowt = make_loss_low_t(apply_train, t_max=T_MAX_LOW)
history_lowt, score_lowt, _ = run_ablation(
    name='Low-t-Only',
    params_init=jax.tree.map(jnp.array, pretrained_params),
    loss_fn=loss_lowt,
    frozen_backbone=False,
    wins_only=False,
)
print(f"Low-t-Only final score: {score_lowt:.4f} (baseline: {baseline_score:.4f})")

## 10. Synthetic Sanity Check

**Purpose:** Rule out that the training infrastructure itself is broken.

Collect PPO rollouts. Perturb 10% of actions randomly. Train with RL to recover
the original actions (reward = 1 if action matches original, 0 otherwise).
This gives dense, clean, achievable reward signal.

- If this also collapses → infrastructure broken OR H1 is very strong
- If this works → the problem is specific to environment reward signal, not the infrastructure


In [ ]:
print("Running synthetic sanity check...")
print("Collecting oracle PPO data for synthetic task...")

# Collect a fixed batch of PPO rollouts as oracle data
rng_synth = jax.random.PRNGKey(999)
rng_synth, env_rng = jax.random.split(rng_synth)
obs_s, es_s = env.reset(env_rng, env_params)
done_s   = jnp.zeros(NUM_ENVS, dtype=bool)
hstate_s = ppo.init_hidden(NUM_ENVS)

rng_synth, rollout_rng = jax.random.split(rng_synth)
_, _, _, _, _, oracle_obs, oracle_acts, oracle_valid, _, _ = collect_rollout(
    es_s, obs_s, done_s, hstate_s, rollout_rng
)
print(f"Oracle data: {oracle_obs.shape[0]} windows")

# Perturb 10% of actions
rng_synth, perturb_rng, rand_rng = jax.random.split(rng_synth, 3)
perturb_mask   = jax.random.bernoulli(perturb_rng, 0.1, shape=oracle_acts.shape)
# FIX: use independent rand_rng (was reusing perturb_rng, producing correlated samples)
random_actions = jax.random.randint(rand_rng, oracle_acts.shape, 0, num_actions)
perturbed_acts = jnp.where(perturb_mask, random_actions, oracle_acts)

# Synthetic reward: fraction of actions matching oracle
def synthetic_reward(model_acts, oracle_acts_ref):
    return jnp.mean((model_acts == oracle_acts_ref).astype(jnp.float32))

# Train to recover: use perturbed_acts as input, oracle_acts as target
# Loss = BC loss on oracle_acts, weighted by whether perturbed_acts differ
synth_state   = create_train_state(net, jax.tree.map(jnp.array, pretrained_params), LR, MAX_GRAD_NORM)
# FIX: add eval_iter so x-axis is tracked explicitly (avoids off-by-one prone range())
synth_history = {'iter': [], 'loss': [], 'recovery_rate': [], 'eval_iter': []}

@jax.jit
def synth_step(state, rng):
    """Train on oracle actions (ground truth recovery task)."""
    def _loss(params):
        # Standard BC loss on oracle actions — no RL, no return weighting.
        # Uses compute_loss directly (consistent with all other loss functions).
        return compute_loss(
            apply_train, params, rng,
            oracle_acts[:BATCH_SIZE], oracle_obs[:BATCH_SIZE], oracle_valid[:BATCH_SIZE],
            num_actions, schedule_fn, schedule_deriv_fn,
            sigma_t=TRAIN_SIGMA, label_smoothing=LABEL_SMOOTHING, advantages=None)[0]
    loss_val, grads = jax.value_and_grad(_loss)(state.params)
    state = state.apply_gradients(grads=grads)
    return state, loss_val

print("Training synthetic recovery task...")
for iteration in range(1, MAX_ITER + 1):
    rng_synth, step_rng, eval_rng = jax.random.split(rng_synth, 3)
    synth_state, loss_val = synth_step(synth_state, step_rng)

    if iteration % 10 == 0:
        synth_history['iter'].append(iteration)
        synth_history['loss'].append(float(loss_val))

    if iteration % EVAL_EVERY == 0:
        eval_info = eval_policy(synth_state.params, eval_rng)
        eval_score = float(eval_info.get('returned_episode_returns', jnp.array(0.0)))
        synth_history['eval_iter'].append(iteration)
        synth_history['recovery_rate'].append(eval_score)
        print(f"  [Synthetic] iter {iteration} | loss={float(loss_val):.4f} | eval_score={eval_score:.4f}")

score_synthetic = synth_history['recovery_rate'][-1] if synth_history['recovery_rate'] else 0.0
print(f"\nSynthetic final score: {score_synthetic:.4f}")
print(f"Interpretation: ", end="")
if score_synthetic > baseline_score - 0.005:
    print("WORKS — infrastructure is fine, problem is specific to RL signal")
elif score_synthetic < baseline_score * 0.5:
    print("FAILS — either infrastructure broken or H1 is very strong")
else:
    print("NEUTRAL — partial recovery")


## 11. Summary and Full Analysis

In [ ]:
results = [
    ('Pretrained (no FT)',  baseline_score),
    ('Baseline RL',         score_baseline_rl),
    ('KL Penalty',          score_kl),
    ('Frozen Backbone',     score_frozen),
    ('BC on Wins',          score_bc),
    ('Low-t Only',          score_lowt),
    ('Synthetic (BC)',      score_synthetic),
]

print('\n' + '='*60)
print(f'{"Method":<25} | {"Score":>10} | {"Delta":>10} | {"Verdict":>12}')
print('='*60)
for name, score in results:
    delta   = score - baseline_score
    flag    = '  ← BASELINE' if name == 'Pretrained (no FT)' else ''
    if name == 'Pretrained (no FT)':
        verdict = 'BASELINE'
    elif delta < -0.005:
        verdict = 'COLLAPSE'
    elif delta > 0.005:
        verdict = 'IMPROVEMENT'
    else:
        verdict = 'NEUTRAL'
    print(f'  {name:<23} | {score:>10.4f} | {delta:>+9.4f} | {verdict:>12}{flag}')
print('='*60)

# --- Gradient alignment analysis ---
print('\n--- Gradient Alignment Analysis ---')
print('(cos_sim < 0 = gradient actively wrong, ~0 = noise, > 0 = useful signal)\n')
all_histories = {
    'Baseline-RL':     history_baseline_rl,
    'KL-Penalty':      history_kl,
    'Frozen-Backbone': history_frozen,
    'BC-on-Wins':      history_bc,
    'Low-t-Only':      history_lowt,
}
for name, hist in all_histories.items():
    if hist['grad_align']:
        mean_align = np.mean(hist['grad_align'])
        final_align = hist['grad_align'][-1]
        trend = '↓' if len(hist['grad_align']) > 1 and hist['grad_align'][-1] < hist['grad_align'][0] else '→'
        print(f'  {name:<23}: mean={mean_align:+.4f}  final={final_align:+.4f}  {trend}')

# --- Representation drift analysis ---
print('\n--- Representation Drift (KL from pretrained) ---')
print('(higher = more drift from pretrained representations)\n')
for name, hist in all_histories.items():
    if hist['repr_drift']:
        mean_drift  = np.mean(hist['repr_drift'])
        final_drift = hist['repr_drift'][-1]
        print(f'  {name:<23}: mean={mean_drift:.6f}  final={final_drift:.6f}')

# --- t-distribution analysis ---
print('\n--- t-Distribution: High-t vs Low-t Gradient Norms ---')
print('(if norm_high >> norm_low: high-t dominates = supports bias hypothesis)\n')
for name, hist in all_histories.items():
    if hist['norm_high_t']:
        ratio = np.mean(hist['norm_high_t']) / (np.mean(hist['norm_low_t']) + 1e-10)
        cos   = np.mean(hist['lowhigh_cos'])
        print(f'  {name:<23}: high/low ratio={ratio:.2f}  low-high alignment={cos:+.4f}')

# --- Paper verdict ---
print('\n--- Paper Verdict ---')
all_collapse = all(s < baseline_score - 0.005
                   for name, s in results[1:-1])  # exclude synthetic
synth_works  = score_synthetic > baseline_score - 0.005

mean_align_baseline = np.mean(history_baseline_rl['grad_align']) if history_baseline_rl['grad_align'] else 0.0

if all_collapse and not synth_works:
    print('  ALL ablations collapse AND synthetic fails.')
    print('  STRONGEST case for H1: fundamental incompatibility with RL fine-tuning.')
elif all_collapse and synth_works:
    print('  ALL ablations collapse BUT synthetic works.')
    print('  Infrastructure is fine. Problem is RL signal from environment interaction.')
    print('  H1 supported: the model cannot improve from environment-generated data.')
else:
    print('  Mixed results — some ablations work. Check individual verdicts above.')

if mean_align_baseline < -0.01:
    print(f'  Gradient alignment = {mean_align_baseline:+.4f}: RL gradient actively points WRONG direction.')
    print('  This PROVES the gradient is not a valid policy gradient surrogate.')
elif abs(mean_align_baseline) < 0.05:
    print(f'  Gradient alignment = {mean_align_baseline:+.4f}: RL gradient is noise.')
    print('  Signal too weak to improve, but not actively harmful.')


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

colors = {
    'Baseline-RL':     '#9E9E9E',
    'KL-Penalty':      '#2196F3',
    'Frozen-Backbone': '#FF7043',
    'BC-on-Wins':      '#4CAF50',
    'Low-t-Only':      '#9C27B0',
}
# all_histories is defined in Cell 29 — not redefined here.

# 1. Eval score over training
ax = axes[0, 0]
ax.axhline(baseline_score, color='black', linestyle='--', linewidth=2,
           label=f'Pretrained ({baseline_score:.4f})', alpha=0.8)
for name, hist in all_histories.items():
    if hist['eval_score']:
        ax.plot(hist['eval_iter'], hist['eval_score'],
                color=colors[name], marker='o', linewidth=2, markersize=4, label=name)
ax.set_title('Eval Score Over Training')
ax.set_xlabel('Iteration'); ax.set_ylabel('Score')
ax.legend(fontsize=7); ax.grid(alpha=0.3)

# 2. Final score bar chart
ax2 = axes[0, 1]
all_names_bar  = ['Pretrained', 'Baseline\nRL', 'KL\nPenalty',
                   'Frozen\nBackbone', 'BC on\nWins', 'Low-t\nOnly', 'Synthetic']
all_scores_bar = [baseline_score, score_baseline_rl, score_kl,
                   score_frozen, score_bc, score_lowt, score_synthetic]
bar_colors_list = ['#607D8B', '#9E9E9E', '#2196F3', '#FF7043', '#4CAF50', '#9C27B0', '#FF9800']
bars = ax2.bar(range(len(all_names_bar)), all_scores_bar, color=bar_colors_list, alpha=0.85)
ax2.axhline(baseline_score, color='black', linestyle='--', alpha=0.5)
ax2.set_xticks(range(len(all_names_bar)))
ax2.set_xticklabels(all_names_bar, fontsize=8)
ax2.set_title('Final Score: All Methods'); ax2.set_ylabel('Score')
ax2.grid(axis='y', alpha=0.3)
for bar, score in zip(bars, all_scores_bar):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
             f'{score:.4f}', ha='center', va='bottom', fontsize=7)

# 3. Gradient alignment over training
ax3 = axes[0, 2]
ax3.axhline(0, color='black', linestyle='--', alpha=0.5, label='Zero (noise)')
for name, hist in all_histories.items():
    if hist['grad_align']:
        ax3.plot(hist['grad_align_iter'], hist['grad_align'],
                 color=colors[name], marker='s', linewidth=1.5, markersize=4, label=name)
ax3.set_title('Gradient Alignment (cos sim with BC gradient)')
ax3.set_xlabel('Iteration'); ax3.set_ylabel('Cosine Similarity')
ax3.legend(fontsize=7); ax3.grid(alpha=0.3)
ax3.set_ylim(-1.1, 1.1)

# 4. Representation drift
ax4 = axes[1, 0]
for name, hist in all_histories.items():
    if hist['repr_drift']:
        ax4.plot(hist['repr_drift_iter'], hist['repr_drift'],
                 color=colors[name], marker='^', linewidth=1.5, markersize=4, label=name)
ax4.set_title('Representation Drift (KL from pretrained)')
ax4.set_xlabel('Iteration'); ax4.set_ylabel('KL Divergence')
ax4.legend(fontsize=7); ax4.grid(alpha=0.3)

# 5. High-t vs Low-t gradient norm ratio
ax5 = axes[1, 1]
for name, hist in all_histories.items():
    if hist['norm_high_t']:
        ratios = [h / (l + 1e-10) for h, l in zip(hist['norm_high_t'], hist['norm_low_t'])]
        ax5.plot(hist['t_analysis_iter'], ratios,
                 color=colors[name], marker='D', linewidth=1.5, markersize=4, label=name)
ax5.axhline(1.0, color='black', linestyle='--', alpha=0.5, label='Equal contribution')
ax5.set_title('High-t / Low-t Gradient Norm Ratio')
ax5.set_xlabel('Iteration'); ax5.set_ylabel('Ratio (>1 = high-t dominates)')
ax5.legend(fontsize=7); ax5.grid(alpha=0.3)

# 6. Synthetic sanity check
ax6 = axes[1, 2]
if synth_history['recovery_rate']:
    # FIX: use eval_iter for x-axis (was range() formula, off-by-one prone)
    ax6.plot(synth_history['eval_iter'],
             synth_history['recovery_rate'],
             color='#FF9800', marker='o', linewidth=2, markersize=5, label='Synthetic BC')
ax6.axhline(baseline_score, color='black', linestyle='--', alpha=0.7,
            label=f'Pretrained ({baseline_score:.4f})')
ax6.set_title('Synthetic Sanity Check (BC on oracle data)')
ax6.set_xlabel('Iteration'); ax6.set_ylabel('Score')
ax6.legend(fontsize=8); ax6.grid(alpha=0.3)

plt.tight_layout()
# FIX: save to FIG_DIR as .pdf (vector format), not cwd .png
import os; os.makedirs(FIG_DIR, exist_ok=True)
_save_path = os.path.join(FIG_DIR, 'ablation_overview.pdf')
plt.savefig(_save_path, bbox_inches='tight')
plt.show()
print(f'Saved: {_save_path}')


In [ ]:
# FIX: route output to OUTPUT_DIR (not cwd). OUTPUT_DIR is set in Cell 34 after
# importing src/ablations/. Fall back to 'notebooks/ablation_results' if needed.
_out_dir = globals().get('OUTPUT_DIR', 'notebooks/ablation_results')
import os; os.makedirs(_out_dir, exist_ok=True)
_results_path = os.path.join(_out_dir, 'craftax_full_results.json')

results_dict = {
    'env': 'Craftax-Classic-Symbolic-v1',
    'max_iter': MAX_ITER,
    'baseline_score':      float(baseline_score),
    'baseline_rl_score':   float(score_baseline_rl),
    'kl_penalty':          float(score_kl),
    'frozen_backbone':     float(score_frozen),
    'bc_on_wins':          float(score_bc),
    'low_t_only':          float(score_lowt),
    'synthetic':           float(score_synthetic),
    'histories': {
        k: {
            hk: [float(v) for v in hv]
            for hk, hv in hist.items()
        }
        for k, hist in {
            'baseline_rl':     history_baseline_rl,
            'kl_penalty':      history_kl,
            'frozen_backbone': history_frozen,
            'bc_on_wins':      history_bc,
            'low_t_only':      history_lowt,
            'synthetic':       synth_history,
        }.items()
    }
}
with open(_results_path, 'w') as f:
    json.dump(results_dict, f, indent=2)
print(f'Saved: {_results_path}')


## Imports from src/ablations/

All new functions for extended ablations are imported from the `src/ablations/` package.
Existing inline definitions above remain for backward-compatibility with Sections 5–9.

In [ ]:
# New imports — src/ablations/ package
# NOTE: sys.path is already set up in Cell 2 via the REPO variable. No duplicate needed.
import yaml
from src.ablations.losses import (
    make_loss_baseline, make_loss_kl, make_loss_bc_wins, make_loss_low_t,
    make_loss_ewc, make_loss_mixed_replay, make_loss_t_curriculum,
    make_loss_entropy_reg, make_loss_token_advantage, make_loss_trust_region,
)
from src.ablations.techniques import compute_ewc_fisher
from src.ablations.diagnostics import (
    compute_gradient_alignment, compute_representation_drift,
    compute_output_kl, compute_per_t_loss, compute_token_entropy,
    compute_collapse_fraction, compute_per_layer_grad_norm,
)
from src.ablations.visualisations import (
    plot_training_dynamics, plot_summary_bars, plot_scatter_diagnostics,
    plot_per_method_deep_dive, plot_t_bin_heatmap, plot_failure_mode_map,
    plot_achievement_bars, make_summary_table, make_correlation_table,
)
from src.ablations.runner import run_ablation_v2

# Load ablation-specific hyperparameters.
with open("../configs/ablations.yaml") as _f:
    ABL_CFG = yaml.safe_load(_f)

OUTPUT_DIR = ABL_CFG.get("ablation_output_dir", "ablation_results")
FIG_DIR   = f"../{OUTPUT_DIR}/figures"
TABLE_DIR = f"../{OUTPUT_DIR}/tables"

import os; os.makedirs(FIG_DIR, exist_ok=True); os.makedirs(TABLE_DIR, exist_ok=True)
print("Ablation config loaded:", ABL_CFG)

## 5b. Extended Ablations

Tests the new RL fine-tuning techniques:
| Ablation | Family | Hypothesis |
|---|---|---|
| EWC | Regularisation | Penalise deviation from pretrained weighted by Fisher info |
| Mixed Replay | Data | Blend online GRPO data with frozen offline PPO windows |
| T-Curriculum | Data | Start low-t only, expand gradually — avoid noisy high-t regime |
| Entropy Reg | Arch/Loss | Entropy bonus maintains action diversity during RL |
| Token Advantage | Arch/Loss | Weight each token by uncertainty — focus on hard positions |
| Trust Region | Training | KL hinge budget prevents policy from drifting too far |

In [ ]:
# ─── 5b.0  Shared setup for extended ablations ────────────────────────────
# Re-use the env, model, and apply_fn objects from Section 2.
# Re-use pretrained_params, apply_train, schedule_fn, schedule_deriv_fn,
# collect_rollout, and run_ablation from the existing sections above.

ALL_HISTORIES_EXT = {}
ALL_RESULTS_EXT   = {}

def _wrap_loss(loss_module_fn):
    """Thin wrapper to match the notebook's (params, rng, acts, obs, valid, adv) signature."""
    return loss_module_fn

EXT_MAX_ITER  = ABL_CFG.get("max_iter",   MAX_ITER)
EXT_EVAL_EVERY = ABL_CFG.get("eval_every", EVAL_EVERY)
EXT_LR         = ABL_CFG.get("lr",         LR)
EXT_BATCH      = ABL_CFG.get("batch_size", BATCH_SIZE)
print(f"Extended ablation settings: max_iter={EXT_MAX_ITER}, eval_every={EXT_EVAL_EVERY}, lr={EXT_LR}, batch={EXT_BATCH}")

In [ ]:
# ─── 5b.1  EWC ────────────────────────────────────────────────────────────
print("\n[EWC] Computing Fisher information matrix...")

# FIX: collect_rollout signature is (env_state, obs, done, hstate, rng)
# not (rng, pretrained_params) as originally written.
rng, ewc_env_rng = jax.random.split(rng)
_obs_ewc, _es_ewc = env.reset(ewc_env_rng, env_params)
_done_ewc   = jnp.zeros(NUM_ENVS, dtype=bool)
_hstate_ewc = ppo.init_hidden(NUM_ENVS)
rng, ewc_roll_rng = jax.random.split(rng)
_, _, _, _, _, _obs_ref, _acts_ref, _valid_ref, _, _ = collect_rollout(
    _es_ewc, _obs_ewc, _done_ewc, _hstate_ewc, ewc_roll_rng
)

rng, ewc_rng = jax.random.split(rng)
fisher = compute_ewc_fisher(
    apply_train, pretrained_params, ewc_rng,
    _obs_ref[:min(128, _obs_ref.shape[0])],
    _acts_ref[:min(128, _acts_ref.shape[0])],
    _valid_ref[:min(128, _valid_ref.shape[0])],
    num_actions, schedule_fn, schedule_deriv_fn,
)
print("[EWC] Fisher computed. Running ablation...")

loss_ewc = make_loss_ewc(
    apply_train, pretrained_params, fisher, num_actions,
    schedule_fn, schedule_deriv_fn,
    ewc_lambda=ABL_CFG["ewc_lambda"],
    plan_horizon=PLAN_HORIZON,
)

# FIX: use run_ablation (Cell 13) not run_ablation_v2 — it closes over env, apply_fn etc.
history_ewc, score_ewc, _ = run_ablation(
    name='EWC',
    params_init=jax.tree.map(jnp.array, pretrained_params),
    loss_fn=loss_ewc,
    frozen_backbone=False,
    wins_only=False,
)
ALL_HISTORIES_EXT['ewc'] = history_ewc
ALL_RESULTS_EXT['ewc'] = {'final_score': score_ewc}
print(f"[EWC] Final eval score: {score_ewc:.4f}")


In [ ]:
# ─── 5b.2  Mixed Replay ──────────────────────────────────────────────────
print("\n[Mixed Replay] Building offline buffer...")

# FIX: collect_rollout signature is (env_state, obs, done, hstate, rng)
rng, mr_env_rng = jax.random.split(rng)
_obs_mr, _es_mr = env.reset(mr_env_rng, env_params)
_done_mr   = jnp.zeros(NUM_ENVS, dtype=bool)
_hstate_mr = ppo.init_hidden(NUM_ENVS)
rng, mr_roll_rng = jax.random.split(rng)
_, _, _, _, _, _offline_obs, _offline_acts, _offline_valid, _, _ = collect_rollout(
    _es_mr, _obs_mr, _done_mr, _hstate_mr, mr_roll_rng
)
_valid_mask      = _offline_valid.astype(bool)
offline_obs_buf  = _offline_obs[_valid_mask]
offline_acts_buf = _offline_acts[_valid_mask]
print(f"[Mixed Replay] Offline buffer: {offline_obs_buf.shape[0]} windows")

loss_mixed = make_loss_mixed_replay(
    apply_train, offline_obs_buf, offline_acts_buf,
    num_actions, schedule_fn, schedule_deriv_fn,
    online_ratio=ABL_CFG["mixed_replay_ratio"],
)

# FIX: use run_ablation (Cell 13) not run_ablation_v2
history_mixed, score_mixed, _ = run_ablation(
    name='MixedReplay',
    params_init=jax.tree.map(jnp.array, pretrained_params),
    loss_fn=loss_mixed,
    frozen_backbone=False,
    wins_only=False,
)
ALL_HISTORIES_EXT['mixed_replay'] = history_mixed
ALL_RESULTS_EXT['mixed_replay'] = {'final_score': score_mixed}
print(f"[Mixed Replay] Final eval score: {score_mixed:.4f}")


In [ ]:
# ─── 5b.3  T-Curriculum ──────────────────────────────────────────────────
print("\n[T-Curriculum] Running ablation...")
loss_curriculum = make_loss_t_curriculum(
    apply_train, num_actions, schedule_fn, schedule_deriv_fn,
    t_curriculum_start=ABL_CFG["t_curriculum_start"],
    t_curriculum_end=ABL_CFG["t_curriculum_end"],
    t_curriculum_steps=ABL_CFG["t_curriculum_steps"],
)

# FIX: use run_ablation (Cell 13) not run_ablation_v2
history_curric, score_curric, _ = run_ablation(
    name='T-Curriculum',
    params_init=jax.tree.map(jnp.array, pretrained_params),
    loss_fn=loss_curriculum,
    frozen_backbone=False,
    wins_only=False,
)
ALL_HISTORIES_EXT['t_curriculum'] = history_curric
ALL_RESULTS_EXT['t_curriculum'] = {'final_score': score_curric}
print(f"[T-Curriculum] Final eval score: {score_curric:.4f}")


In [ ]:
# ─── 5b.4  Entropy Regularisation ────────────────────────────────────────
print("\n[Entropy Reg] Running ablation...")
loss_entropy = make_loss_entropy_reg(
    apply_train, num_actions, schedule_fn, schedule_deriv_fn,
    entropy_coef=ABL_CFG["entropy_coef"],
    plan_horizon=PLAN_HORIZON,
)

# FIX: use run_ablation (Cell 13) not run_ablation_v2
history_entropy, score_entropy, _ = run_ablation(
    name='EntropyReg',
    params_init=jax.tree.map(jnp.array, pretrained_params),
    loss_fn=loss_entropy,
    frozen_backbone=False,
    wins_only=False,
)
ALL_HISTORIES_EXT['entropy_reg'] = history_entropy
ALL_RESULTS_EXT['entropy_reg'] = {'final_score': score_entropy}
print(f"[Entropy Reg] Final eval score: {score_entropy:.4f}")


In [ ]:
# ─── 5b.5  Token-Level Advantage ─────────────────────────────────────────
print("\n[Token Advantage] Running ablation...")
loss_token_adv = make_loss_token_advantage(
    apply_train, num_actions, schedule_fn, schedule_deriv_fn,
)

# FIX: use run_ablation (Cell 13) not run_ablation_v2
history_token_adv, score_token_adv, _ = run_ablation(
    name='TokenAdvantage',
    params_init=jax.tree.map(jnp.array, pretrained_params),
    loss_fn=loss_token_adv,
    frozen_backbone=False,
    wins_only=False,
)
ALL_HISTORIES_EXT['token_advantage'] = history_token_adv
ALL_RESULTS_EXT['token_advantage'] = {'final_score': score_token_adv}
print(f"[Token Advantage] Final eval score: {score_token_adv:.4f}")


In [ ]:
# ─── 5b.6  Trust Region ──────────────────────────────────────────────────
print("\n[Trust Region] Running ablation...")
loss_trust = make_loss_trust_region(
    apply_train, pretrained_params, num_actions, schedule_fn, schedule_deriv_fn,
    trust_region_kl=ABL_CFG["trust_region_kl"],
    kl_penalty_coef=ABL_CFG["trust_region_penalty_coef"],
    plan_horizon=PLAN_HORIZON,
)

# FIX: use run_ablation (Cell 13) not run_ablation_v2
history_trust, score_trust, _ = run_ablation(
    name='TrustRegion',
    params_init=jax.tree.map(jnp.array, pretrained_params),
    loss_fn=loss_trust,
    frozen_backbone=False,
    wins_only=False,
)
ALL_HISTORIES_EXT['trust_region'] = history_trust
ALL_RESULTS_EXT['trust_region'] = {'final_score': score_trust}
print(f"[Trust Region] Final eval score: {score_trust:.4f}")


## 5b Summary — Extended Ablations

In [ ]:
# Merge all results (original + extended) for unified comparison.
ALL_HISTORIES_COMBINED = {
    "baseline_rl":     history_baseline_rl,
    "kl_penalty":      history_kl,
    "frozen_backbone": history_frozen,
    "bc_wins":         history_bc,
    "low_t":           history_lowt,
    **ALL_HISTORIES_EXT,
}
ALL_RESULTS_COMBINED = {
    "baseline_rl":     {"final_score": score_baseline_rl,  "pretrained_score": baseline_score},
    "kl_penalty":      {"final_score": score_kl,           "pretrained_score": baseline_score},
    "frozen_backbone": {"final_score": score_frozen,       "pretrained_score": baseline_score},
    "bc_wins":         {"final_score": score_bc,           "pretrained_score": baseline_score},
    "low_t":           {"final_score": score_lowt,         "pretrained_score": baseline_score},
    **{k: {**v, "pretrained_score": baseline_score} for k, v in ALL_RESULTS_EXT.items()},
}

# Compute gradient alignment and representation drift for each method.
_probe = None
for method, history in ALL_HISTORIES_COMBINED.items():
    grads_list  = history.get("grads",      [])
    aligns_list = history.get("grad_align", [])
    drifts_list = history.get("repr_drift", [])
    # Use last available values as "final" state.
    ALL_RESULTS_COMBINED[method]["final_grad_align"] = (
        float(aligns_list[-1]) if aligns_list else 0.0
    )
    ALL_RESULTS_COMBINED[method]["final_drift"] = (
        float(drifts_list[-1]) if drifts_list else 0.0
    )

print("All ablation results merged.")
print(f"Methods: {list(ALL_RESULTS_COMBINED.keys())}")
fig_dynamics = plot_training_dynamics(
    ALL_HISTORIES_COMBINED, baseline_score, save_dir=FIG_DIR
)
plt.show()

## 12. Correlation Analysis

Identifies which diagnostic metrics are most predictive of final eval score
across all methods.  Strong correlates are candidates for early-stopping
criteria and monitoring dashboards.

In [ ]:
# ─── Section 12: Diagnostic vs. Score Correlation ─────────────────────────
corr_df = make_correlation_table(ALL_RESULTS_COMBINED, target_key="final_score")
print("Pearson / Spearman correlation of diagnostics with final eval score:")
if len(corr_df) > 0:
    print(corr_df)
else:
    print("(Insufficient data — run with multiple seeds or more methods for correlations)")

# Scatter plots.
fig_scatter = plot_scatter_diagnostics(ALL_RESULTS_COMBINED, save_dir=FIG_DIR)
plt.show()

## 13. Failure Mode Taxonomy

Classifies each run into one of four failure modes based on the joint pattern
of representation drift and gradient alignment:

| Quadrant | Drift | Alignment | Failure mode |
|---|---|---|---|
| High drift, low alignment  | High | Negative | Catastrophic forgetting |
| High drift, high alignment | High | Positive | Mode collapse |
| Low drift, low alignment   | Low  | Negative | Gradient conflict |
| Low drift, high alignment  | Low  | Positive | No learning / Success |

In [ ]:
# ─── Section 13: Failure Mode Map ─────────────────────────────────────────
fig_fail = plot_failure_mode_map(ALL_RESULTS_COMBINED, save_dir=FIG_DIR)
plt.show()

# Per-method deep-dive panels for all runs.
for method, history in ALL_HISTORIES_COMBINED.items():
    _has_diagnostics = any(
        k in history for k in ["token_entropy", "output_kl", "per_layer_grad_norm"]
    )
    if _has_diagnostics:
        fig_dd = plot_per_method_deep_dive(method, history, save_dir=FIG_DIR)
        plt.show()

# T-bin heatmap for all methods.
fig_tbin = plot_t_bin_heatmap(ALL_HISTORIES_COMBINED, save_dir=FIG_DIR)
plt.show()

# Achievement bars (if achievement data is present).
_has_ach = any("achievements" in h for h in ALL_HISTORIES_COMBINED.values())
if _has_ach:
    fig_ach = plot_achievement_bars(ALL_HISTORIES_COMBINED, save_dir=FIG_DIR)
    plt.show()

# Full summary bar chart.
fig_bars = plot_summary_bars(ALL_RESULTS_COMBINED, save_dir=FIG_DIR)
plt.show()

# Summary table (LaTeX + CSV).
latex_table = make_summary_table(ALL_RESULTS_COMBINED, save_dir=f"../{OUTPUT_DIR}", latex=True)
print("\nLaTeX Summary Table:")
print(latex_table)

In [ ]:
# ─── Save results to disk for generate_report.py ──────────────────────────
# FIX: use stdlib json instead of orjson (orjson is not in environment.yaml).
import json as _json

def _convert(obj):
    """Recursively convert JAX/numpy arrays to Python primitives for JSON serialisation."""
    import numpy as _np
    if isinstance(obj, _np.ndarray):
        return obj.tolist()
    if isinstance(obj, dict):
        return {k: _convert(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_convert(v) for v in obj]
    try:
        return float(obj)
    except (TypeError, ValueError):
        return str(obj)

_serialisable_results   = _convert({'_pretrained_score': baseline_score, **ALL_RESULTS_COMBINED})
_serialisable_histories = _convert(ALL_HISTORIES_COMBINED)

_res_path = f"../{OUTPUT_DIR}/results.json"
_his_path = f"../{OUTPUT_DIR}/histories.json"

with open(_res_path, 'w') as _f:
    _json.dump(_serialisable_results, _f, indent=2, default=float)
with open(_his_path, 'w') as _f:
    _json.dump(_serialisable_histories, _f, indent=2, default=float)

print(f'Results saved to: {_res_path}')
print(f'Histories saved to: {_his_path}')
print('\nRun the standalone report script with:')
print(f'  python notebooks/generate_report.py --results_dir {OUTPUT_DIR}')
